# Python Threading — A Structured Lecture for Engineering Students

**Course:** Advanced Python Programming  
**Topic:** Concurrent Execution with the `threading` Module  

---

> *"Concurrency is not parallelism, and parallelism is not always the answer.  
> Understanding the difference — and knowing when each applies — is the hallmark  
> of a well-trained engineer."*

---


## Learning Objectives

By the end of this lecture, you will be able to:

1. Explain the difference between a **process** and a **thread**.
2. Identify **I/O-bound** and **CPU-bound** workloads and choose the correct concurrency model.
3. Create and manage threads using Python's `threading` module.
4. Describe the **thread life cycle** and the role of each state.
5. Detect and eliminate **race conditions** using synchronisation primitives (`Lock`, `Semaphore`).
6. Articulate the effect of the **Global Interpreter Lock (GIL)** on multi-threaded Python code.
7. Use `ThreadPoolExecutor` from `concurrent.futures` to manage a pool of worker threads efficiently.


## 1 · Background: Concurrency and Parallelism

Before examining threads, it is important to distinguish two commonly confused concepts:

| Concept | Definition | Example |
|---------|------------|---------|
| **Concurrency** | Multiple tasks make progress by sharing CPU time (interleaving). | One cook preparing multiple dishes by switching attention. |
| **Parallelism** | Multiple tasks execute simultaneously on separate CPU cores. | Multiple cooks each preparing a separate dish at the same time. |

Python's `threading` module provides **concurrency** through interleaving, not true parallelism  
(due to the Global Interpreter Lock, which we will examine in §9).


## 2 · Process vs Thread

### 2.1 Process
A **process** is an independent instance of a running program.  
The operating system allocates each process its own:
- Virtual memory space
- File descriptors
- CPU registers and call stack

Processes are **isolated** by design; one process cannot directly access another's memory.

### 2.2 Thread
A **thread** (sometimes called a *lightweight process*) is a unit of execution that lives  
**inside** a process. All threads of the same process share:
- The same heap memory (global and heap variables)
- Open file handles
- Network connections

This shared-memory model makes inter-thread communication fast and easy —  
but it also introduces the risk of **race conditions** (see §7).

### 2.3 When to use threads
- **I/O-bound tasks** (network calls, disk reads, database queries): threads are effective  
  because a waiting thread releases the GIL and allows other threads to proceed.  
- **CPU-bound tasks** (numerical computation, image processing): use `multiprocessing`  
  instead, because the GIL prevents true parallel execution of Python bytecode.


## 3 · The `threading` Module — Core API Reference

Python's standard library provides the `threading` module for high-level thread management.

### 3.1 `threading.Thread` — the fundamental class

```
threading.Thread(group=None, target=None, name=None, args=(), kwargs={}, daemon=None)
```

| Parameter | Role |
|-----------|------|
| `target` | Callable (function) executed in the new thread |
| `name` | Optional human-readable name; defaults to `"Thread-N"` |
| `args` | Positional arguments passed to `target` |
| `kwargs` | Keyword arguments passed to `target` |
| `daemon` | `True` → thread is a daemon (exits when the main program exits) |

#### Key methods

| Method | Purpose |
|--------|---------|
| `.start()` | Schedules the thread; internally calls `.run()` |
| `.run()` | Contains the thread logic; override in subclasses |
| `.join(timeout=None)` | Blocks the calling thread until this thread terminates |
| `.is_alive()` | Returns `True` if the thread is still executing |
| `.name` | Get or set the thread name |
| `.daemon` | `True` if this is a daemon thread |

### 3.2 Module-level functions

| Function | Returns |
|----------|---------|
| `threading.current_thread()` | The `Thread` object representing the caller |
| `threading.main_thread()` | The `Thread` object for the main thread |
| `threading.active_count()` | Number of currently alive `Thread` objects |
| `threading.enumerate()` | List of all currently alive `Thread` objects |


## 4 · Step 1 — Sequential Execution (Establishing a Baseline)

We always start with a **sequential implementation** and measure its performance  
before introducing concurrency. This gives us a meaningful baseline for comparison.

The function below simulates an I/O-bound operation (e.g., a network call) by  
sleeping for one second. Sequential execution calls it ten times, giving a total  
run time of approximately **10 seconds**.


In [ ]:
from time import sleep, time

def fetch_data(task_id: int) -> None:
    """Simulate an I/O-bound operation (e.g., a network request)."""
    print(f"  [Task {task_id}] Starting I/O operation...")
    sleep(1)                          # blocks for 1 second, simulating I/O wait
    print(f"  [Task {task_id}] I/O complete.")

# ── Sequential execution ──────────────────────────────────────────────────────
print("=== Sequential Execution ===")
start = time()

for i in range(10):
    fetch_data(i)

elapsed = time() - start
print(f"\nTotal time (sequential): {elapsed:.2f} s")
print("Expected: ~10 s  (10 tasks × 1 s each)")


## 5 · Step 2 — Creating Your First Thread

Now we offload one task to a separate thread.  
Notice the following:

1. We **create** the thread with `threading.Thread(target=..., args=[...])`.
2. We **start** the thread with `.start()` — this is non-blocking; the main thread continues immediately.
3. We **join** the thread with `.join()` — this makes the main thread wait for the worker to finish.

This introduces the fundamental three-step pattern:  
**Create → Start → Join**


In [ ]:
from time import sleep, time
import threading

def fetch_data(task_id: int) -> None:
    """Simulate an I/O-bound operation."""
    print(f"  [Task {task_id}] Starting I/O operation...")
    sleep(1)
    print(f"  [Task {task_id}] I/O complete.")

# ── Single-threaded execution ─────────────────────────────────────────────────
print("=== One Thread ===")
start = time()

worker = threading.Thread(target=fetch_data, args=[0])   # Step 1: Create
worker.start()                                            # Step 2: Start
worker.join()                                             # Step 3: Wait for completion

elapsed = time() - start
print(f"\nTotal time (1 thread): {elapsed:.2f} s")


## 6 · Step 3 — Multiple Threads Running Concurrently

We now launch **ten threads**, one per task, all started before any `join()` is called.  
Because all ten threads sleep concurrently, the total wall-clock time collapses to  
approximately **1 second** instead of 10 — a 10× speed-up for this I/O-bound workload.

**Observation:** Start all threads first, then join them all.  
Do **not** interleave start and join, or you will serialize the threads.


In [ ]:
from time import sleep, time
import threading

def fetch_data(task_id: int) -> None:
    print(f"  [Task {task_id}] Starting I/O operation...")
    sleep(1)
    print(f"  [Task {task_id}] I/O complete.")

print("=== Ten Concurrent Threads ===")
start = time()

# Create all thread objects
threads = [threading.Thread(target=fetch_data, args=[i]) for i in range(10)]

# Start every thread BEFORE joining any
for t in threads:
    t.start()

# Now wait for all threads to finish
for t in threads:
    t.join()

elapsed = time() - start
print(f"\nTotal time (10 threads): {elapsed:.2f} s")
print("Expected: ~1 s  (all tasks overlap in time)")


## 7 · Step 4 — Passing Arguments and Returning Results

### 7.1 Passing arguments
Arguments are supplied via the `args` (positional) or `kwargs` (keyword) parameters of `Thread`.

### 7.2 Returning values
Threads do **not** return values directly through `join()`.  
The idiomatic approach is to store the result in a shared mutable container  
(e.g., a list or dictionary) before the thread exits.


In [ ]:
import threading
import time

results = {}   # shared dictionary — each thread writes to its own key

def compute_square(n: int) -> None:
    """Compute the square of n and store the result."""
    time.sleep(0.1)                   # simulate a short I/O pause
    results[n] = n * n

numbers = [2, 3, 8, 9]

threads = [threading.Thread(target=compute_square, args=[n]) for n in numbers]

for t in threads:
    t.start()
for t in threads:
    t.join()

print("Input  :", numbers)
print("Squares:", [results[n] for n in numbers])


## 8 · Step 5 — The Thread Life Cycle

A `Thread` object transitions through the following states during its lifetime:

```
NEW  ──(start())──▶  RUNNABLE  ──(scheduler)──▶  RUNNING
                                                    │
                    ◀──(I/O wait / sleep())──  BLOCKED/WAITING
                                                    │
                                              TERMINATED
```

| State | Description |
|-------|-------------|
| **New** | Thread object created; `start()` not yet called |
| **Runnable** | `start()` called; thread is queued for CPU time |
| **Running** | Thread is currently executing on a CPU core |
| **Blocked / Waiting** | Thread is sleeping, waiting on I/O, or blocked on a lock |
| **Terminated** | `run()` method has returned; thread cannot be restarted |

The snippet below logs each state transition so you can observe the life cycle directly.


In [ ]:
import threading
import time

def lifecycle_demo(name: str, work_time: float) -> None:
    print(f"  [{name}] RUNNING — thread is executing")
    time.sleep(work_time)            # enters BLOCKED/WAITING
    print(f"  [{name}] RUNNING — resumed after sleep")
    # function returns → TERMINATED

t1 = threading.Thread(target=lifecycle_demo, args=("Thread-A", 1.0), name="Thread-A")
t2 = threading.Thread(target=lifecycle_demo, args=("Thread-B", 0.5), name="Thread-B")

print(f"t1 is alive (NEW):      {t1.is_alive()}")  # False — not started yet

t1.start(); t2.start()
print(f"t1 is alive (RUNNING):  {t1.is_alive()}")  # True — started

t1.join();  t2.join()
print(f"t1 is alive (TERMINATED): {t1.is_alive()}")  # False — finished

print("\nActive thread count:", threading.active_count())
print("Current thread:", threading.current_thread().name)
print("Main thread:", threading.main_thread().name)


## 9 · Step 6 — Daemon Threads

### Definition
A **daemon thread** is a background thread whose lifecycle is subordinate to the main thread.  
When the main thread (and all non-daemon threads) terminate, the Python runtime  
automatically kills any surviving daemon threads — **without** waiting for them to finish.

### When to use daemon threads
- Background monitoring tasks (heartbeats, log flushers)
- Tasks that have no meaningful cleanup requirement
- Threads that should not keep the program alive

### How to create a daemon thread
Set `daemon=True` in the constructor, **before** calling `start()`.

| Thread type | Programme exit behaviour |
|-------------|--------------------------|
| Non-daemon (default) | Main thread waits for the thread to finish |
| Daemon | Main thread exits immediately; daemon is killed |


In [ ]:
import threading
import time

def background_monitor() -> None:
    """Daemon thread: logs a heartbeat every 0.5 s."""
    while True:
        print(f"  [Monitor] heartbeat — alive threads: {threading.active_count()}")
        time.sleep(0.5)

def main_task() -> None:
    """Non-daemon thread: does real work for 2 seconds."""
    print("  [Worker] started")
    time.sleep(2)
    print("  [Worker] finished")

monitor = threading.Thread(target=background_monitor, name="Monitor", daemon=True)
worker  = threading.Thread(target=main_task,         name="Worker")

monitor.start()    # daemon — will be killed when main exits
worker.start()     # non-daemon — main will wait for this one

worker.join()      # main thread waits only for the worker
print("\nMain thread: all non-daemon threads done. Programme will exit now.")
# The daemon 'Monitor' thread is killed automatically.


## 10 · Step 7 — Useful `threading` Module Functions

The examples below demonstrate `threading.active_count()`, `threading.enumerate()`,  
`threading.current_thread()`, and `threading.main_thread()` in context.


In [ ]:
import threading
import time

def timed_task(name: str, duration: float) -> None:
    time.sleep(duration)

# Launch several short-lived threads
threads = [
    threading.Thread(target=timed_task, args=[f"T{i}", 0.3 * i], name=f"Worker-{i}")
    for i in range(1, 6)
]
for t in threads:
    t.start()

# Snapshot of live threads immediately after starting them
print(f"Active thread count  : {threading.active_count()}")
print(f"Current thread       : {threading.current_thread().name}")
print(f"Main thread          : {threading.main_thread().name}")
print("Live threads         :", [t.name for t in threading.enumerate()])

for t in threads:
    t.join()
print("\nAll worker threads terminated.")


## 11 · Step 8 — Race Conditions: The Hidden Danger of Shared State

### The problem
When two or more threads read and modify the **same variable** without coordination,  
the final value depends on the unpredictable order in which the CPU interleaves  
their instructions. This is called a **race condition**.

### Anatomy of a race condition
Consider `balance += amount`. At the machine level this is three steps:
1. **READ** `balance` from memory into a register.
2. **ADD** `amount` to the register.
3. **WRITE** the register back to `balance`.

If a second thread reads `balance` between steps 1 and 3 of the first thread,  
it sees the **old** value and both threads eventually write conflicting results.

### Demonstration
The code below deposits and withdraws one unit one million times each.  
The expected final balance is **200** (unchanged), but without synchronisation  
the result is non-deterministic.


In [ ]:
import threading

INITIAL_BALANCE = 200
balance = INITIAL_BALANCE

def deposit(amount: int, iterations: int) -> None:
    global balance
    for _ in range(iterations):
        balance += amount       # NOT atomic — subject to race conditions

def withdraw(amount: int, iterations: int) -> None:
    global balance
    for _ in range(iterations):
        balance -= amount       # NOT atomic — subject to race conditions

deposit_thread  = threading.Thread(target=deposit,  args=[1, 1_000_000])
withdraw_thread = threading.Thread(target=withdraw, args=[1, 1_000_000])

deposit_thread.start()
withdraw_thread.start()

deposit_thread.join()
withdraw_thread.join()

print(f"Expected balance : {INITIAL_BALANCE}")
print(f"Actual balance   : {balance}")
print(f"Discrepancy      : {balance - INITIAL_BALANCE}  ← caused by the race condition")


## 12 · Step 9 — Thread Synchronisation with `Lock`

### What is a lock?
A `threading.Lock` is a **mutual exclusion** primitive.  
At any point in time, at most **one** thread may hold the lock.

- `lock.acquire()` — blocks until the lock is free, then claims it.
- `lock.release()` — releases the lock so another thread may acquire it.

Using the `with` statement is strongly preferred over manual `acquire`/`release`  
calls: it guarantees that the lock is always released, even if an exception occurs.

### Corrected bank-account example
We wrap the read-modify-write sequence inside a `with lock:` block,  
making the critical section **atomic** with respect to other threads.


In [ ]:
import threading

INITIAL_BALANCE = 200
balance = INITIAL_BALANCE
lock = threading.Lock()          # one lock guards the shared variable

def deposit(amount: int, iterations: int) -> None:
    global balance
    for _ in range(iterations):
        with lock:                # acquire lock → execute → release lock
            balance += amount

def withdraw(amount: int, iterations: int) -> None:
    global balance
    for _ in range(iterations):
        with lock:
            balance -= amount

deposit_thread  = threading.Thread(target=deposit,  args=[1, 1_000_000])
withdraw_thread = threading.Thread(target=withdraw, args=[1, 1_000_000])

deposit_thread.start()
withdraw_thread.start()

deposit_thread.join()
withdraw_thread.join()

print(f"Expected balance : {INITIAL_BALANCE}")
print(f"Actual balance   : {balance}")
print("Result is deterministic — the lock prevents race conditions.")


## 13 · Step 10 — Controlling Concurrency with `Semaphore`

A **semaphore** is a generalised lock that permits up to **N** threads to enter  
a critical section simultaneously.  It maintains an internal counter:

- `acquire()` decrements the counter; blocks if the counter is already 0.
- `release()` increments the counter, potentially unblocking a waiting thread.

### Use case
A web-scraping application must not open more than 3 concurrent HTTP connections  
to avoid overloading the server. A `Semaphore(3)` enforces this limit.


In [ ]:
import threading
import time

MAX_CONCURRENT = 3
semaphore = threading.Semaphore(MAX_CONCURRENT)

def fetch_url(url_id: int) -> None:
    with semaphore:               # at most MAX_CONCURRENT threads inside here
        print(f"  [URL {url_id:02d}] Fetching... (active: {MAX_CONCURRENT - semaphore._value})")
        time.sleep(1)             # simulate network latency
        print(f"  [URL {url_id:02d}] Done.")

print(f"Fetching 8 URLs with a concurrency limit of {MAX_CONCURRENT}\n")
threads = [threading.Thread(target=fetch_url, args=[i]) for i in range(8)]

for t in threads:
    t.start()
for t in threads:
    t.join()

print("\nAll URLs fetched.")


## 14 · Step 11 — The Global Interpreter Lock (GIL)

### What is the GIL?
CPython (the reference Python interpreter) uses a mutex called the  
**Global Interpreter Lock** that ensures only **one thread executes Python bytecode  
at a time**, even on a multi-core machine.

### Why does the GIL exist?
CPython's memory management (reference counting) is not thread-safe without a  
global lock. The GIL was chosen as a pragmatic solution when Python was designed.

### Practical consequences

| Workload | Effect of GIL | Recommendation |
|----------|---------------|----------------|
| **I/O-bound** (network, disk) | Minimal — the GIL is released during I/O syscalls | `threading` is effective |
| **CPU-bound** (computation) | Severe — threads cannot run Python in parallel | Use `multiprocessing` or C extensions that release the GIL |

### Demonstration: I/O-bound vs CPU-bound with threads


In [ ]:
import threading
import time

# ── I/O-bound workload ────────────────────────────────────────────────────────
def io_task(task_id: int) -> None:
    time.sleep(1)                 # GIL is released during sleep → true concurrency

print("=== I/O-bound: 4 threads ===")
start = time.time()
threads = [threading.Thread(target=io_task, args=[i]) for i in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(f"  Wall time: {time.time() - start:.2f} s  (expected ~1 s — threads overlap)\n")

# ── CPU-bound workload ────────────────────────────────────────────────────────
def cpu_task(n: int) -> int:
    return sum(i * i for i in range(n))

print("=== CPU-bound: 4 threads ===")
start = time.time()
threads = [threading.Thread(target=cpu_task, args=[5_000_000]) for _ in range(4)]
for t in threads: t.start()
for t in threads: t.join()
print(f"  Wall time: {time.time() - start:.2f} s  (expected ≥ sequential time — GIL serialises execution)")


## 15 · Step 12 — `ThreadPoolExecutor` from `concurrent.futures`

Managing raw `Thread` objects becomes unwieldy when the number of tasks is large  
or when we need to **collect return values**. Python 3.2 introduced the  
`concurrent.futures` module, which provides a higher-level interface.

### 15.1 `ThreadPoolExecutor`
- Maintains a **pool** of reusable worker threads (avoids repeated thread creation overhead).
- `executor.submit(fn, *args)` — schedules one task; returns a `Future` object.
- `future.result()` — blocks until the task completes and returns its return value.
- `executor.map(fn, iterable)` — applies `fn` to every element; returns results in order.
- `as_completed(futures)` — yields `Future` objects as they complete (not in submission order).

### 15.2 Single task with `submit`


In [ ]:
from time import sleep, time
from concurrent.futures import ThreadPoolExecutor

def fetch_data(task_id: int) -> str:
    print(f"  [Task {task_id}] Starting...")
    sleep(1)
    return f"Result from task {task_id}"

print("=== submit() — single task ===")
start = time()

with ThreadPoolExecutor(max_workers=4) as executor:
    future = executor.submit(fetch_data, 42)
    print(f"  Future done? {future.done()}")   # False — still running
    result = future.result()                   # blocks until done
    print(f"  Result: {result}")

print(f"Elapsed: {time() - start:.2f} s\n")


In [ ]:
from time import sleep, time
from concurrent.futures import ThreadPoolExecutor, as_completed

def fetch_data(task_id: int) -> str:
    sleep(1)
    return f"Result from task {task_id}"

print("=== submit() — 10 tasks, results in completion order ===")
start = time()

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(fetch_data, i) for i in range(10)]

    for completed in as_completed(futures):
        print(" ", completed.result())

print(f"Elapsed: {time() - start:.2f} s  (expected ~2 s with 5 workers)\n")


In [ ]:
from time import sleep, time
from concurrent.futures import ThreadPoolExecutor

def fetch_data(task_id: int) -> str:
    sleep(1)
    return f"Result from task {task_id}"

print("=== map() — results in submission order ===")
start = time()

with ThreadPoolExecutor(max_workers=5) as executor:
    # map() preserves the order of the input iterable
    for result in executor.map(fetch_data, range(10)):
        print(" ", result)

print(f"Elapsed: {time() - start:.2f} s\n")


## 16 · Step 13 — Worked Exercise: Squares and Cubes

**Problem:** Given a list of numbers, compute the square and cube of every element.  
Demonstrate the performance difference between sequential execution and threading.

### 16.1 Sequential version (baseline)


In [ ]:
import time

def calculate_squares(numbers: list) -> None:
    print("Computing squares...")
    for n in numbers:
        time.sleep(0.2)           # simulate I/O per element
        print(f"  {n}² = {n * n}")

def calculate_cubes(numbers: list) -> None:
    print("Computing cubes...")
    for n in numbers:
        time.sleep(0.2)
        print(f"  {n}³ = {n ** 3}")

data = [2, 3, 8, 9]

start = time.time()
calculate_squares(data)
calculate_cubes(data)
print(f"\nSequential time: {time.time() - start:.2f} s  (expected ~1.6 s)")


### 16.2 Concurrent version using `threading`

Both functions now run **simultaneously**. Because each sleeps for 0.2 s per element,  
the concurrent version takes roughly half the time of the sequential version.


In [ ]:
import time
import threading

def calculate_squares(numbers: list) -> None:
    print("Computing squares...")
    for n in numbers:
        time.sleep(0.2)
        print(f"  {n}² = {n * n}")

def calculate_cubes(numbers: list) -> None:
    print("Computing cubes...")
    for n in numbers:
        time.sleep(0.2)
        print(f"  {n}³ = {n ** 3}")

data = [2, 3, 8, 9]

start = time.time()

t_sq   = threading.Thread(target=calculate_squares, args=[data])
t_cube = threading.Thread(target=calculate_cubes,   args=[data])

t_sq.start()
t_cube.start()

t_sq.join()
t_cube.join()

print(f"\nConcurrent time: {time.time() - start:.2f} s  (expected ~0.8 s — ~2× speed-up)")


## 17 · Step 14 — Subclassing `Thread` (Object-Oriented Approach)

When a thread requires its own state or more complex initialisation,  
you may subclass `threading.Thread` and override `__init__` and `run`.

This is the **object-oriented** alternative to the function-based approach  
demonstrated in previous steps.


In [ ]:
import threading
import time

class TimedWorker(threading.Thread):
    """
    A thread that executes a given number of timed iterations.

    Parameters
    ----------
    name     : Descriptive name for this worker.
    steps    : Number of work iterations to perform.
    interval : Sleep duration (seconds) between iterations.
    """

    def __init__(self, name: str, steps: int, interval: float) -> None:
        super().__init__(name=name)
        self.steps    = steps
        self.interval = interval

    def run(self) -> None:
        for step in range(1, self.steps + 1):
            time.sleep(self.interval)
            print(f"  [{self.name}] Step {step}/{self.steps} complete")

# Instantiate two workers with different configurations
worker_a = TimedWorker(name="Worker-A", steps=3, interval=0.4)
worker_b = TimedWorker(name="Worker-B", steps=5, interval=0.2)

worker_a.start()
worker_b.start()

worker_a.join()
worker_b.join()

print("Both workers have finished.")


## 18 · Summary and Engineering Best Practices

### 18.1 Summary of concepts covered

| Step | Concept | Key takeaway |
|------|---------|--------------|
| 1 | Sequential baseline | Always measure before optimising |
| 2–3 | `Thread` create / start / join | The fundamental three-step pattern |
| 4 | Returning results | Use shared containers; `Thread` has no return value |
| 5 | Thread life cycle | New → Runnable → Running → Blocked → Terminated |
| 6 | Daemon threads | Use for background tasks that need no cleanup |
| 7 | Module functions | `active_count`, `enumerate`, `current_thread` |
| 8 | Race conditions | Shared mutable state + multiple threads = undefined behaviour |
| 9 | `Lock` | Mutual exclusion; prefer `with lock:` over manual acquire/release |
| 10 | `Semaphore` | Limit concurrency to N simultaneous threads |
| 11 | GIL | I/O-bound → threading; CPU-bound → multiprocessing |
| 12 | `ThreadPoolExecutor` | High-level API; use `submit`, `map`, `as_completed` |

### 18.2 Best practices

1. **Prefer `ThreadPoolExecutor` over raw threads** for production code — it handles  
   pool sizing, error propagation, and resource cleanup.

2. **Never access shared mutable state without a lock.** Even simple `x += 1`  
   is not atomic in Python.

3. **Minimise the critical section.** Hold the lock only as long as necessary  
   to reduce contention.

4. **Avoid nested locks without a strict ordering** — this is the classic source  
   of **deadlocks**.

5. **Use daemon threads sparingly.** Data written by a daemon thread may be lost  
   if the process exits before the write is flushed.

6. **Profile before parallelising.** Concurrency adds complexity; confirm it  
   actually improves performance before adopting it.


## 19 · Practice Exercises

Work through the following exercises independently. Start each one sequentially,  
measure the baseline, then introduce threading and compare the results.

---

**Exercise 1 — File Downloader Simulation**  
Write a function `simulate_download(url_id, size_mb)` that sleeps for `size_mb * 0.1` seconds.  
Download 10 files sequentially, then concurrently using `ThreadPoolExecutor`.  
Record and compare the wall-clock time for each approach.

---

**Exercise 2 — Thread-Safe Counter**  
Create a shared counter initialised to 0. Launch 5 threads, each incrementing  
the counter 100 000 times. Without a lock, observe the incorrect final value.  
Then add a `Lock` and verify the result is always 500 000.

---

**Exercise 3 — Producer–Consumer with a Queue**  
Use `queue.Queue` (which is inherently thread-safe) to implement a producer that  
generates 20 integer items and three consumer threads that each print and process  
items as they arrive. Ensure graceful shutdown when all items are consumed.

---

**Exercise 4 — Semaphore Rate Limiter**  
Simulate a database connection pool with a maximum of 4 concurrent connections.  
Launch 12 tasks; each holds a "connection" for 0.5 seconds before releasing it.  
Use a `Semaphore(4)` and verify that at most 4 tasks are active at any moment.

---

**Exercise 5 — Comparing Threading vs Multiprocessing**  
Implement a CPU-bound function (e.g., summing squares up to 10 000 000).  
Run it with (a) sequential execution, (b) 4 `Thread` workers,  
and (c) 4 `Process` workers from `multiprocessing`.  
Tabulate the results and explain the differences in terms of the GIL.
